In [ ]:
import os
#os.environ["OMP_NUM_THREADS"] = "1"
#os.environ["OPENBLAS_NUM_THREADS"] = "1"
#os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np

import numpy as np
import matplotlib.pyplot as plt
import igraph as ig
import json


%load_ext autoreload
%autoreload 2
from src.simulation import run_replicas, simulate

In [ ]:
import time
from itertools import product

# Test with your actual production parameters
RANKER_CONFIGS = [...]  # your full list
N_REPLICAS = 100        # your actual target

times = []
for ranker_cfg in RANKER_CONFIGS:
    INFO["Ranker"] = ranker_cfg
    tic = time.time()
    run_replicas(INFO, n_replicas=5)  # small sample
    elapsed = time.time() - tic
    per_replica = elapsed / 5
    times.append((ranker_cfg['rule'], per_replica))
    print(f"{ranker_cfg['rule']}: {per_replica:.2f}s/replica  → {per_replica * N_REPLICAS / 60:.1f} min for {N_REPLICAS} replicas")

worst = max(t for _, t in times)
print(f"\nWorst case job time: {worst * N_REPLICAS / 60:.1f} min → request {worst * N_REPLICAS / 60 * 1.5:.0f} min")

In [ ]:
import os
import time
from itertools import product

RANKER_CONFIGS = [
    {'rule': 'Random'},
    {'rule': 'Closest'},
    {'rule': 'Engagement',         'alpha': 1.0},
    {'rule': 'Narrative',          'target_opinion': 0.5},
    {'rule': 'Evil',               'target_opinion': 0.5},
    {'rule': 'Diverse_Engagement'},
    {'rule': 'User_Success',       'alpha': 1.0},
    {'rule': 'Personalization',    'alpha': 1.0},
]

OD_CONFIGS = [
    {'model': 'BCM',            'epsilon': 0.2, 'mu': 0.1},
    {'model': 'BCM_HK',         'epsilon': 0.2, 'mu': 0.1},
    {'model': 'BCM_negative',   'epsilon_1': 0.2, 'epsilon_2': 0.6, 'mu': 0.1},
    {'model': 'BCM_asymmetric', 'epsilon': 0.2, 'mu_1': 0.2, 'mu_2': 0.05},
]

NOISE_VALUES = [0.0, 0.2]
KPOST_VALUES = [1, 5]
N_REPLICAS   = 100

times = []

for od_cfg, ranker_cfg, noise, k_posts in product(OD_CONFIGS, RANKER_CONFIGS, NOISE_VALUES, KPOST_VALUES):
    INFO = {
        "General": {"seed": 42},
        "Graph": {"type": "ER", "n": 1000, "p": 0.1},
        "OD": od_cfg,
        "Ranker": ranker_cfg,
        "Simulation_details": {"n_steps": 10000, "convergence_window": 50, "convergence_delta": 1e-6},
        "k_posts": k_posts,
        "post_history": 50,
        "noise": noise,
    }

    label = f"{od_cfg['model']} + {ranker_cfg['rule']} | noise={noise} k={k_posts}"

    tic = time.time()
    for rep in range(2):
        simulate(INFO, seed=42 + rep)
    per_replica = (time.time() - tic) / 2

    estimated_min = per_replica * N_REPLICAS * 2 / 60  # x2 safety margin
    times.append((label, per_replica, estimated_min))
    print(f"{label}  →  {per_replica:.2f}s/replica  →  {estimated_min:.1f} min")

worst_label, worst_per_replica, worst_min = max(times, key=lambda x: x[1])
print(f"\nWorst case: {worst_label}")
print(f"  {worst_per_replica:.2f}s/replica  →  request {worst_min:.0f} min per job")

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import time
from itertools import product

RANKER_CONFIGS = [
    {'rule': 'Random'},
    {'rule': 'Closest'},
    {'rule': 'Engagement',         'alpha': 1.0},
    {'rule': 'Narrative',          'target_opinion': 0.5},
    {'rule': 'Evil',               'target_opinion': 0.5},
    {'rule': 'Diverse_Engagement'},
    {'rule': 'User_Success',       'alpha': 1.0},
    {'rule': 'Personalization',    'alpha': 1.0},
]

OD_CONFIGS = [
    {'model': 'BCM',            'epsilon': 0.2, 'mu': 0.1},
    {'model': 'BCM_HK',         'epsilon': 0.2, 'mu': 0.1},
    {'model': 'BCM_negative',   'epsilon_1': 0.2, 'epsilon_2': 0.6, 'mu': 0.1},
    {'model': 'BCM_asymmetric', 'epsilon': 0.2, 'mu_1': 0.2, 'mu_2': 0.05},
]

NOISE_VALUES = [0.0, 0.2]
KPOST_VALUES = [1, 5]
N_REPLICAS   = 100

times = []

for od_cfg, ranker_cfg, noise, k_posts in product(OD_CONFIGS, RANKER_CONFIGS, NOISE_VALUES, KPOST_VALUES):
    INFO = {
        "General": {"seed": 42},
        "Graph": {"type": "ER", "n": 1000, "p": 0.1},
        "OD": od_cfg,
        "Ranker": ranker_cfg,
        "Simulation_details": {"n_steps": 10000, "convergence_window": 50, "convergence_delta": 1e-6},
        "k_posts": k_posts,
        "post_history": 50,
        "noise": noise,
    }

    label = f"{od_cfg['model']} + {ranker_cfg['rule']} | noise={noise} k={k_posts}"

    tic = time.time()
    for rep in range(2):
        simulate(INFO, seed=42 + rep)
    per_replica = (time.time() - tic) / 2

    estimated_min = per_replica * N_REPLICAS * 2 / 60  # x2 safety margin
    times.append((label, per_replica, estimated_min))
    print(f"{label}  →  {per_replica:.2f}s/replica  →  {estimated_min:.1f} min")

worst_label, worst_per_replica, worst_min = max(times, key=lambda x: x[1])
print(f"\nWorst case: {worst_label}")
print(f"  {worst_per_replica:.2f}s/replica  →  request {worst_min:.0f} min per job")

In [ ]:
info = {
    "Graph": {"type": "ER", "n": 1000, "p": 0.01},
    "OD": {"model": "BCM", "epsilon": 0.3, "mu": 0.1},
    "Ranker": {"rule": "Random"},
    "Simulation_details": {"n_steps": 500, "convergence_window": 50, "convergence_delta": 1e-4},
    "post_history": 50,
    "k_posts": 3,
}

result = simulate(info)

print(f"Convergence step: {result['convergence_step']}")
print(f"Final polarization: {result['pol'][result['convergence_step']]:.6f}")
print(f"Final mean opinion: {result['final_opinions'].mean():.6f}")
print(f"Final opinions (first 10): {result['final_opinions'][:10]}")

In [ ]:
# print the list of keys in the result dictionary
print("Result keys:", result.keys())

In [ ]:
import matplotlib.pyplot as plt
info = {
    "Graph": {"type": "ER", "n": 1000, "p": 0.01},
    "OD": {"model": "BCM", "epsilon": 0.3, "mu": 0.1},
    "Ranker": {"rule": "Random"},
    "Simulation_details": {"n_steps": 500, "convergence_window": 50, "convergence_delta": 1e-4},
    "post_history": 50,
    "k_posts": 3,
}

result = simulate(info)
plt.plot(result["opinions"])

In [ ]:
info = {
    "Graph": {"type": "ER", "n": 100, "p": 0.1},
    "OD": {"model": "BCM", "epsilon": 0.3, "mu": 0.1},
    "Ranker": {"rule": "Random"},
    "Simulation_details": {"n_steps": 500, "convergence_window": 50, "convergence_delta": 1e-4},
    "post_history": 50,
    "k_posts": 3,
}

result = simulate(info)

print(f"Convergence step: {result['convergence_step']}")
print(f"Final polarization: {result['pol'][result['convergence_step']]:.6f}")
print(f"Final mean opinion: {result['final_opinions'].mean():.6f}")
print(f"Final opinions (first 10): {result['final_opinions'][:10]}")

In [ ]:
import cProfile
import pstats

info = {
    "Graph": {"type": "ER", "n": 1000, "p": 0.01},
    "OD": {"model": "BCM", "epsilon": 0.3, "mu": 0.1},
    "Ranker": {"rule": "Random"},
    "Simulation_details": {"n_steps": 500, "convergence_window": 50, "convergence_delta": 1e-4},
    "post_history": 50,
    "k_posts": 3,
}

profiler = cProfile.Profile()
profiler.enable()
result = simulate(info)
profiler.disable()

print(f"Convergence step: {result['convergence_step']}")
print(f"Final polarization: {result['pol'][result['convergence_step']]:.6f}")
print(f"Final mean opinion: {result['final_opinions'].mean():.6f}")
print(f"Final opinions (first 10): {result['final_opinions'][:10]}")

stats = pstats.Stats(profiler)
stats.sort_stats('cumulative')
stats.print_stats(20)  # top 20 functions

In [ ]:
import cProfile
import pstats

info = {
    "Graph": {"type": "ER", "n": 1000, "p": 0.01},
    "OD": {"model": "BCM", "epsilon": 0.3, "mu": 0.1},
    "Ranker": {"rule": "User_Success", "alpha": 1.0},
    "Simulation_details": {"n_steps": 500, "convergence_window": 50, "convergence_delta": 1e-4},
    "post_history": 50,
    "k_posts": 3,
}

profiler = cProfile.Profile()
profiler.enable()
result = simulate(info)
profiler.disable()

print(f"Convergence step: {result['convergence_step']}")
print(f"Final polarization: {result['pol'][result['convergence_step']]:.6f}")
print(f"Final mean opinion: {result['final_opinions'].mean():.6f}")
print(f"Final opinions (first 10): {result['final_opinions'][:10]}")

stats = pstats.Stats(profiler)
stats.sort_stats('cumulative')
stats.print_stats(20)  # top 20 functions